# **3일차 팀 프로젝트: 테이블 데이터 조회 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 CSV 테이블 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. Text2SQL 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [35]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

**TODO: 팀에서 준비한 CSV 파일 경로를 입력하세요**

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [36]:
import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "iot_common_guidelines": "../datasets/1_parent_iot_common_guidelines.csv",
    "home_iot_controls": "../datasets/2_child_home_iot_controls.csv",
    "stm32_debug_topics": "../datasets/3_child_stm32_debug_topics.csv",
    "esp32h2_features": "../datasets/4_child_esp32h2_features.csv"
}


# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path)
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 iot_common_guidelines 테이블

행 수: 15
컬럼: ['guideline_id', 'principle_id', 'principle_name', 'lifecycle_stage', 'guideline_code', 'guideline_name', 'page_start', 'domain', 'description', 'design_focus', 'security_focus', 'easy_explanation']

첫 5개 행:
   guideline_id  principle_id                     principle_name  \
0             1             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
1             2             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
2             3             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
3             4             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
4             5             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   

  lifecycle_stage guideline_code                guideline_name  page_start  \
0           설계·개발            G01     IoT 장치 특성을 고려한 보안 서비스 경량화          20   
1           설계·개발            G02  접근권한 관리·인증·종단간 통신 보안·데이터 암호화          31   
2           설계·개발            G03         소프트웨어·하드웨어 보안기술 적용 검토          33   
3           설계·개발            G

## 2. 데이터 탐색 및 통계

**TODO: 팀 데이터에 맞는 탐색 쿼리를 작성하세요**

In [37]:
for table_name, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {table_name} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    df.info()

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()

    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # 고유값 개수
    print("\n[고유값 개수]")
    for col in df.columns:
        print(f"{col}: {df[col].nunique()}개")

    # 카테고리별 데이터 분포
    print("\n[카테고리 분포]")
    for col in df.select_dtypes(include=['object']).columns:
        print(f"\n[{col}]")
        print(df[col].value_counts())


📊 iot_common_guidelines 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   guideline_id      15 non-null     int64
 1   principle_id      15 non-null     int64
 2   principle_name    15 non-null     str  
 3   lifecycle_stage   15 non-null     str  
 4   guideline_code    15 non-null     str  
 5   guideline_name    15 non-null     str  
 6   page_start        15 non-null     int64
 7   domain            15 non-null     str  
 8   description       15 non-null     str  
 9   design_focus      15 non-null     str  
 10  security_focus    15 non-null     str  
 11  easy_explanation  15 non-null     str  
dtypes: int64(3), str(9)
memory usage: 8.7 KB

[결측치]
결측치 없음

[고유값 개수]
guideline_id: 15개
principle_id: 7개
principle_name: 7개
lifecycle_stage: 3개
guideline_code: 15개
guideline_name: 15개
page_start: 15개
domain: 15개
description: 15개
design_focus

C:\Users\khm35\AppData\Local\Temp\ipykernel_10744\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
C:\Users\khm35\AppData\Local\Temp\ipykernel_10744\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

## 3. Supabase PostgreSQL 연결

In [38]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['1_parent_iot_common_guidelines', '2_child_home_iot_controls', '3_child_stm32_debug_topics', '4_child_esp32h2_features', 'categories', 'departments', 'forms', 'office_floors', 'organizations', 'topics']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. **반드시 테이블 관계 순서대로 업로드** (부모 → 자식)
4. Foreign Key 에러 발생 시 순서를 재확인

## 5. 업로드 확인 및 스키마 탐색

In [39]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f"SELECT * FROM {table} LIMIT 3")
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE "1_parent_iot_common_guidelines" (
	guideline_id BIGINT, 
	principle_id BIGINT, 
	principle_name TEXT, 
	lifecycle_stage TEXT, 
	guideline_code TEXT, 
	guideline_name TEXT, 
	page_start BIGINT, 
	domain TEXT, 
	description TEXT, 
	design_focus TEXT, 
	security_focus TEXT, 
	easy_explanation TEXT
)

/*
3 rows from 1_parent_iot_common_guidelines table:
guideline_id	principle_id	principle_name	lifecycle_stage	guideline_code	guideline_name	page_start	domain	description	design_focus	security_focus	easy_explanation
1	1	정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계	설계·개발	G01	IoT 장치 특성을 고려한 보안 서비스 경량화	20	하드웨어_설계	프로세서 성능, 메모리, 입출력장치, 소비전력 등 장치 자원 수준을 고려하여 필요한 보안 기능을 경량화해 구현	MCU 성능, 메모리, 입출력장치, 소비전력 등 기기 자원을 먼저 확인	기기 성능을 넘지 않는 범위에서 필요한 보안 기능을 적용	기기의 성능과 전력에 맞춰 하드웨어와 보안 기능의 크기를 정하는 기준
2	1	정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계	설계·개발	G02	접근권한 관리·인증·종단간 통신 보안·데이터 암호화	31	인증_통신_암호화	IoT 서비스 환경에 맞는 접근권한, 인증, 통신 보호, 데이터 암호화 방안을 제공	기기와 외부 장치·서버가 연결되는 통신 경로를 확인	허가된 사용자·장치만 접근하게 하고 통신과 데이터를

## 6. SQL 쿼리 테스트

**TODO: 팀 데이터에 맞는 SQL 쿼리를 작성하여 테스트하세요**

In [40]:
# TODO: 기본 조회 쿼리 작성
# ESP32-H2의 하드웨어·보안 기능 통합 조회
# 부모: 1_parent_iot_common_guidelines
# 자식: 4_child_esp32h2_features

query = """
SELECT
    e.feature_name,
    e.feature_category,
    e.module_or_interface,
    e.hardware_or_security,
    e.design_use,
    e.page_start,
    g.guideline_name
FROM "4_child_esp32h2_features" AS e
INNER JOIN "1_parent_iot_common_guidelines" AS g
    ON e.parent_guideline_id = g.guideline_id
WHERE e.hardware_or_security IN ('하드웨어', '보안', '통합')
ORDER BY
    CASE e.hardware_or_security
        WHEN '하드웨어' THEN 1
        WHEN '통합' THEN 2
        WHEN '보안' THEN 3
        ELSE 4
    END,
    e.page_start;
"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    e.feature_name,
    e.feature_category,
    e.module_or_interface,
    e.hardware_or_security,
    e.design_use,
    e.page_start,
    g.guideline_name
FROM "4_child_esp32h2_features" AS e
INNER JOIN "1_parent_iot_common_guidelines" AS g
    ON e.parent_guideline_id = g.guideline_id
WHERE e.hardware_or_security IN ('하드웨어', '보안', '통합')
ORDER BY
    CASE e.hardware_or_security
        WHEN '하드웨어' THEN 1
        WHEN '통합' THEN 2
        WHEN '보안' THEN 3
        ELSE 4
    END,
    e.page_start;


결과:
[('System and Memory', '메모리 구조', 'ROM / HP SRAM / LP SRAM / External Flash', '하드웨어', '메모리 설계', 142, 'IoT 장치 특성을 고려한 보안 서비스 경량화'), ('IO MUX and GPIO Matrix', '입출력 설계', 'GPIO / IO MUX', '하드웨어', '핀 기능·신호 라우팅 설계', 217, 'IoT 장치 특성을 고려한 보안 서비스 경량화'), ('GPIO Hysteresis', '신호 안정성', 'GPIO', '하드웨어', '입력 노이즈 대응', 227, 'IoT 장치 특성을 고려한 보안 서비스 경량화'), ('GPIO Power Supply Management', '전원·저전력 설계', 'VDDPST1 / VDDPST2 / VDDA_PMU/VBAT', '하드웨어', 'GPIO 전원 도메인·Sleep Wake 설계', 228, 'IoT 장치 특성을 고려

In [41]:
# TODO: JOIN 쿼리 작성
# 부모 테이블과 자식 3개 테이블을 연결하여 하드웨어·보안 데이터 통합 조회

join_query = """
SELECT
    g.guideline_name,
    '홈가전 IoT 보안가이드' AS source_type,
    h.control_name AS item_name,
    h.hardware_or_security AS item_type,
    h.page_start
FROM home_iot_controls h
INNER JOIN iot_common_guidelines g
    ON h.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'STM32 디버깅 가이드' AS source_type,
    s.topic_name AS item_name,
    s.hardware_or_security AS item_type,
    s.page_start
FROM stm32_debug_topics s
INNER JOIN iot_common_guidelines g
    ON s.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'ESP32-H2 기술 매뉴얼' AS source_type,
    e.feature_name AS item_name,
    e.hardware_or_security AS item_type,
    e.page_start
FROM esp32h2_features e
INNER JOIN iot_common_guidelines g
    ON e.parent_guideline_id = g.guideline_id

ORDER BY guideline_name, source_type, page_start;
"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    g.guideline_name,
    '홈가전 IoT 보안가이드' AS source_type,
    h.control_name AS item_name,
    h.hardware_or_security AS item_type,
    h.page_start
FROM home_iot_controls h
INNER JOIN iot_common_guidelines g
    ON h.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'STM32 디버깅 가이드' AS source_type,
    s.topic_name AS item_name,
    s.hardware_or_security AS item_type,
    s.page_start
FROM stm32_debug_topics s
INNER JOIN iot_common_guidelines g
    ON s.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'ESP32-H2 기술 매뉴얼' AS source_type,
    e.feature_name AS item_name,
    e.hardware_or_security AS item_type,
    e.page_start
FROM esp32h2_features e
INNER JOIN iot_common_guidelines g
    ON e.parent_guideline_id = g.guideline_id

ORDER BY guideline_name, source_type, page_start;


결과:
쿼리 실행 오류: (psycopg2.errors.UndefinedTable) relation "home_iot_controls" does not exist
LINE 8: FROM home_iot_controls h
     

In [42]:
# TODO: 집계(Aggregation) 쿼리 작성
# 예시: GROUP BY, COUNT 등 사용

aggregation_query = """
SELECT
    g.guideline_name,
    COUNT(DISTINCT h.home_control_id) AS home_iot_count,
    COUNT(DISTINCT s.stm32_topic_id) AS stm32_count,
    COUNT(DISTINCT e.esp32_feature_id) AS esp32_count
FROM "1_parent_iot_common_guidelines" g
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = g.guideline_id
LEFT JOIN "3_child_stm32_debug_topics" s
    ON s.parent_guideline_id = g.guideline_id
LEFT JOIN "4_child_esp32h2_features" e
    ON e.parent_guideline_id = g.guideline_id
GROUP BY g.guideline_name
ORDER BY home_iot_count DESC;
"""

print("실행 쿼리:")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    g.guideline_name,
    COUNT(DISTINCT h.home_control_id) AS home_iot_count,
    COUNT(DISTINCT s.stm32_topic_id) AS stm32_count,
    COUNT(DISTINCT e.esp32_feature_id) AS esp32_count
FROM "1_parent_iot_common_guidelines" g
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = g.guideline_id
LEFT JOIN "3_child_stm32_debug_topics" s
    ON s.parent_guideline_id = g.guideline_id
LEFT JOIN "4_child_esp32h2_features" e
    ON e.parent_guideline_id = g.guideline_id
GROUP BY g.guideline_name
ORDER BY home_iot_count DESC;


결과:
[('접근권한 관리·인증·종단간 통신 보안·데이터 암호화', 4, 0, 1), ('다양한 하드웨어 보안기법 적용', 4, 10, 6), ('소프트웨어 취약점 점검 및 보안패치 방안 구현', 2, 1, 0), ('소프트웨어·하드웨어 보안기술 적용 검토', 2, 1, 7), ('시큐어코딩 적용', 1, 0, 0), ('개인정보보호정책 및 보호조치 마련', 1, 0, 0), ('민감정보 보호', 1, 0, 1), ('안전한 보안 프로토콜 및 파라미터 설정', 1, 0, 1), ('취약점 분석 및 보안패치 배포', 1, 0, 0), ('로그기록 저장·관리', 1, 0, 0), ('침입탐지 및 모니터링', 0, 0, 0), ('Secure by Default 적용', 0, 0, 1), ('민감정보 운영정책 투명성 보장', 0, 0, 0), ('보안취약점 및 보호조치 공지', 0, 0,

## 7. Text2SQL 함수 구현

**TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요**

In [43]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    system_prompt = f"""
당신은 IoT 디바이스의 하드웨어 설계와 보안 데이터를 다루는 SQL 전문가입니다.
사용자의 자연어 질문을 PostgreSQL SELECT 쿼리로 변환하세요.

데이터베이스 스키마:
{db.table_info}

<데이터베이스 설명>

- "1_parent_iot_common_guidelines":
  IoT 공통 보안 가이드의 상위 기준 테이블입니다.
  guideline_id가 기본 식별자이며,
  principle_name, lifecycle_stage, guideline_name, domain, description 등의 정보를 포함합니다.

- "2_child_home_iot_controls":
  홈·가전 IoT 보안가이드의 세부 보안·하드웨어 점검 항목입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.
  control_name, control_category, applicable_scope,
  hardware_or_security, implementation_summary, page_start 등의 정보를 포함합니다.

- "3_child_stm32_debug_topics":
  STM32 MCU의 하드웨어·디버깅·보안 기능 정보입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.
  topic_name, section_code, topic_type, interface_or_tool,
  hardware_or_security, applicable_scope, page_start 등의 정보를 포함합니다.

- "4_child_esp32h2_features":
  ESP32-H2의 하드웨어 및 보안 기능 정보입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.
  feature_name, feature_category, module_or_interface,
  hardware_or_security, design_use, page_start 등의 정보를 포함합니다.

</데이터베이스 설명>

<테이블 관계>

"1_parent_iot_common_guidelines".guideline_id
    ← "2_child_home_iot_controls".parent_guideline_id

"1_parent_iot_common_guidelines".guideline_id
    ← "3_child_stm32_debug_topics".parent_guideline_id

"1_parent_iot_common_guidelines".guideline_id
    ← "4_child_esp32h2_features".parent_guideline_id

</테이블 관계>

규칙:
- PostgreSQL 문법을 사용하세요.
- SELECT 쿼리만 생성하세요. INSERT, UPDATE, DELETE, DROP, ALTER는 금지합니다.
- 테이블명이 숫자로 시작하므로 테이블명은 반드시 큰따옴표(" ")로 감싸세요.
- 부모와 자식 테이블을 연결할 때 parent_guideline_id = guideline_id 조건으로 JOIN하세요.
- 하드웨어 관련 질문은 hardware_or_security, feature_category, module_or_interface,
  topic_type, interface_or_tool, design_use 등을 우선 활용하세요.
- 보안 관련 질문은 hardware_or_security, control_category, guideline_name,
  domain, description 등을 활용하세요.
- '하드웨어', '보안', '통합'을 구분해야 할 경우 hardware_or_security 컬럼을 사용하세요.
- ESP32-H2 관련 질문은 "4_child_esp32h2_features"를 우선 사용하세요.
- STM32 관련 질문은 "3_child_stm32_debug_topics"를 우선 사용하세요.
- 홈캠, 도어락, 스마트TV 등 홈·가전 IoT 제품 보안 질문은
  "2_child_home_iot_controls"를 우선 사용하세요.
- 공통 보안 원칙이나 상위 가이드 기준이 필요한 경우
  "1_parent_iot_common_guidelines"와 JOIN하세요.
- 여러 문서의 데이터를 함께 비교하는 질문은 UNION ALL 또는 JOIN을 적절히 사용하세요.
- 페이지 관련 질문은 page_start 컬럼을 사용하세요.
- 결과가 너무 많을 가능성이 있으면 필요한 경우 LIMIT을 사용하세요.
- SQL 코드만 반환하세요.
- 설명은 반환하지 마세요.
- 코드 블록(```) 없이 순수 SQL만 반환하세요.
- 반드시 세미콜론(;)으로 끝내세요.

사용 가능한 SQL 문법:
- JOIN (INNER, LEFT, RIGHT, FULL)
- UNION, UNION ALL
- GROUP BY, HAVING
- COUNT, SUM, AVG, MIN, MAX
- 서브쿼리
- WHERE, ORDER BY, LIMIT
- CTE (WITH 절)
- 윈도우 함수
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")

✓ Text2SQL 함수 준비 완료


## 8. Text2SQL 테스트

**TODO: 팀 데이터에 맞는 자연어 질문으로 테스트하세요**

In [44]:
# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "IoT 기기를 만들 때 부품은 어떻게 연결하고, 해킹을 막으려면 어떤 보안 기능을 같이 써야 하나요?"

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = text_to_sql(question, db)
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")

질문: IoT 기기를 만들 때 부품은 어떻게 연결하고, 해킹을 막으려면 어떤 보안 기능을 같이 써야 하나요?


생성된 SQL:
SELECT 
    g.guideline_id,
    g.guideline_code,
    g.guideline_name,
    g.domain,
    g.description,
    g.design_focus,
    g.security_focus,
    g.easy_explanation
FROM "1_parent_iot_common_guidelines" g
WHERE g.domain IN ('하드웨어_설계', '하드웨어_보안통합')
ORDER BY g.guideline_id;


실행 결과:
[(1, 'G01', 'IoT 장치 특성을 고려한 보안 서비스 경량화', '하드웨어_설계', '프로세서 성능, 메모리, 입출력장치, 소비전력 등 장치 자원 수준을 고려하여 필요한 보안 기능을 경량화해 구현', 'MCU 성능, 메모리, 입출력장치, 소비전력 등 기기 자원을 먼저 확인', '기기 성능을 넘지 않는 범위에서 필요한 보안 기능을 적용', '기기의 성능과 전력에 맞춰 하드웨어와 보안 기능의 크기를 정하는 기준'), (3, 'G03', '소프트웨어·하드웨어 보안기술 적용 검토', '하드웨어_보안통합', '소프트웨어 보안기술과 하드웨어 보안기술을 함께 검토하고 안전성이 검증된 보안기술을 적용', '메인 MCU와 하드웨어 보안 기능 또는 보안 모듈의 적용 가능성을 검토', '검증된 소프트웨어·하드웨어 보안기술을 함께 적용', 'MCU 설계와 보안 기능을 따로 보지 않고 같이 적용하는 기준')]


## 9. 완전한 Text2SQL 시스템 (SQL 실행 + 자연어 답변)

**TODO: 답변 생성 프롬프트를 팀 데이터에 맞게 수정하세요**

In [45]:
def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    system_prompt = """
당신은 IoT 기기의 하드웨어 설계와 보안을 함께 도와주는 쉬운 설명 전문가입니다.
SQL 조회 결과를 바탕으로 사용자가 실제로 IoT 기기를 만들 때 참고할 수 있도록 답변하세요.

특히 사용자가
"스마트 도어락 만들어줘",
"스마트 홈 기기를 어떻게 만들어요?",
"이런 IoT 기기는 어떻게 제작하나요?"
처럼 제작 방법을 질문하면 단순히 데이터 이름을 나열하지 마세요.

다음 순서로 쉽게 설명하세요.

1. 필요한 하드웨어
   - 어떤 부품이나 기능이 필요한지
   - 각 부품이 무슨 역할을 하는지

2. 연결 및 구성
   - 센서, MCU, 통신 기능 등을 어떤 용도로 연결하는지
   - SQL 결과에서 확인 가능한 범위까지만 설명

3. 보안 설정
   - 외부 연결 포트 보호
   - 디버깅 기능 보호
   - 인증 및 접근 제한
   - 데이터 암호화와 같은 보안 방법
   - 각각 왜 필요한지 쉬운 말로 설명

4. 최종적으로
   - 하드웨어 설계와 보안을 같이 고려했을 때 무엇을 확인해야 하는지 간단히 정리

답변 규칙:
- SQL 결과에 있는 내용을 중심으로 답변하세요.
- SQL 결과에 없는 구체적인 회로 값, 핀 번호, 부품 모델은 만들어내지 마세요.
- 전문용어를 최대한 줄이세요.
- 전문용어가 필요하면 바로 쉬운 뜻을 붙이세요.
  예: JTAG(기기 내부를 점검하는 연결 통로)
- 기능 이름만 나열하지 말고 "어디에 쓰는지"를 설명하세요.
- 보안 기능은 "무엇을 막기 위한 것인지" 설명하세요.
- 하드웨어와 보안을 따로 떼어 설명하지 말고 서로 연결해서 설명하세요.
- 초보자가 읽어도 이해할 수 있는 쉬운 한국어를 사용하세요.
- 한 문장을 짧게 작성하세요.
- SQL이나 데이터베이스라는 표현은 최종 답변에서 굳이 언급하지 마세요.
- 정보가 부족한 부분은 "현재 자료에서는 구체적인 방법까지 확인하기 어렵습니다."라고 안내하세요.
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

✓ 완전한 Text2SQL 시스템 준비 완료


In [46]:
from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "스마트 도어락 제작 방법 알려줘"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: 스마트 도어락 제작 방법 알려줘


[1] SQL 생성 중...
    SELECT 
  p.guideline_id,
  p.guideline_code,
  p.guideline_name,
  p.lifecycle_stage,
  p.domain,
  p.description,
  c.home_control_id,
  c.control_name,
  c.control_category,
  c.applicable_scope,
  c.hardware_or_security,
  c.implementation_summary,
  c.easy_explanation
FROM "1_parent_iot_common_guidelines" p
JOIN "2_child_home_iot_controls" c
  ON c.parent_guideline_id = p.guideline_id
WHERE c.applicable_scope ILIKE '%도어락%'
   OR c.control_name ILIKE '%도어락%'
   OR c.description ILIKE '%도어락%'
   OR c.easy_explanation ILIKE '%도어락%'
   OR p.guideline_name ILIKE '%도어락%'
   OR p.description ILIKE '%도어락%'
ORDER BY p.guideline_id, c.home_control_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...


답변:


스마트 도어락을 만들 때는 **잠금 기능**만 보는 것보다,  
**직접 만져지는 물리 공격**까지 같이 막는 설계가 중요합니다.  
이번 결과에서 확인되는 핵심은 **“외부 조작 확인 및 분해 방지 메커니즘”**입니다.

## 1. 필요한 하드웨어
현재 자료에서 도어락 제작에 직접 연결되는 하드웨어는 **물리적 보안 구조**입니다.

- **외부 조작을 감지하거나 막는 구조**
  - 누가 도어락을 억지로 열거나 흔드는지 확인하는 데 씁니다.
- **분해 방지 구조**
  - 기기를 뜯어서 내부를 직접 건드리는 일을 막습니다.
- **Tamper Proofing(무단 개봉 방지)**
  - 본체를 열면 흔적이 남거나, 이상 행동을 감지하도록 만드는 뜻입니다.
  - 도어락처럼 손이 닿는 제품에 특히 필요합니다.

즉, 스마트 도어락은 센서나 통신만 넣는 것이 아니라,  
**기기 자체가 쉽게 열리지 않게 만드는 외형과 내부 구조**가 필요합니다.

## 2. 연결 및 구성
현재 자료에서는 **센서, MCU(기기를 제어하는 중심칩), 통신 방식** 같은 구체적 연결은 확인되지 않습니다.  
그래서 구체적인 배선이나 회로 구성까지는 말씀드리기 어렵습니다.

다만 자료 기준으로 보면, 연결과 구성에서 중요한 방향은 다음입니다.

- **도어락 본체가 물리적으로 뜯기지 않게 구성**
  - 내부 부품이 쉽게 노출되지 않도록 합니다.
- **외부에서 손댄 흔적을 확인할 수 있게 구성**
  - 억지 개봉이나 분해 시도를 알아차리기 쉽게 만듭니다.

현재 자료에서는 구체적인 방법까지 확인하기 어렵습니다.  
그래서 센서 종류나 통신 연결 방식은 추가 자료가 있어야 설명할 수 있습니다.

## 3. 보안 설정
이번 결과에서 가장 중요한 부분은 **하드웨어 보안**입니다.  
스마트 도어락은 외부 네트워크보다도, 먼저 **직접 손대는 공격**을 막아야 합니다.

### 외부 연결 포트 보호
- 외부에서 내부로 바로 들어가는 포트는 함부로 열리지 않게 해야 합니다.
- 이유는 누군가 포트를 통해 내부 설정을 바꾸거나 잠금을 우회할 수 있기 때문입니다.

### 디버깅 기능 보호
- 디버깅(기기 내부를 점검하는 기능)은 제조나 점검 때만 필요합니다.
- 제품을 팔고 나면 일반 사용자가 못 쓰게 막아야 합니다.
- 이유는 이 기능이 열려 있으면 내부 정보를 보고 조작할 수 있기 때문입니다.

### 인증 및 접근 제한
- 기기 내부에 들어오는 작업을 아무나 못 하게 제한해야 합니다.
- 이유는 분해 후 임의 조작을 막기 위해서입니다.
- 도어락은 특히 열림 상태를 바꾸는 명령이 중요하므로 더 엄격해야 합니다.

### 데이터 암호화
- 펌웨어나 코드 암호화는 내부 프로그램을 그대로 읽거나 복사하지 못하게 합니다.
- 이유는 제품 구조나 동작 방식을 빼가서 역공학(기기를 뜯어 원리를 분석하는 것)하는 일을 막기 위해서입니다.

### 역공학 방지
- 하드웨어 보안기법을 넣어 내부 동작을 쉽게 분석하지 못하게 합니다.
- 이유는 도어락의 잠금 로직이나 인증 방식을 알아내면 위험하기 때문입니다.

## 4. 최종적으로 무엇을 확인해야 하나
스마트 도어락은 다음을 같이 봐야 합니다.

- **본체가 쉽게 뜯기지 않는가**
- **억지 개봉이나 분해 시도를 알 수 있는가**
- **외부 포트와 디버깅 통로가 잠겨 있는가**
- **내부 코드와 동작 방식이 쉽게 노출되지 않는가**
- **물리 보안과 내부 보안을 함께 설계했는가**

정리하면,  
이번 자료에서 확인되는 스마트 도어락의 핵심은 **“도어락은 손으로 만질 수 있는 제품이므로, 분해 방지와 무단 개봉 감지 같은 물리 보안이 꼭 필요하다”**는 점입니다.  

현재 자료에서는 센서, 통신, 제어칩까지는 구체적으로 확인되지 않아서,  
실제 제작 회로까지는 더 많은 정보가 있어야 설명할 수 있습니다.

## 10. 다양한 질문으로 테스트

**TODO: 최소 5개 이상의 다양한 질문으로 테스트하세요**

In [47]:
# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "스마트 도어락을 만들 때 어떤 하드웨어와 보안 기능이 필요한가요?",
    "ESP32-H2에서 부품 연결에 쓰는 기능과 해킹을 막는 보안 기능을 같이 알려주세요.",
    "STM32에서 기기 내부를 보호하기 위해 사용할 수 있는 기능은 무엇인가요?",
    "IoT 공통 보안 가이드별로 연결된 홈가전, STM32, ESP32-H2 항목이 각각 몇 개인가요?",
    "하드웨어와 관련된 기능들을 페이지 순서대로 보여주세요."
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")


질문: 스마트 도어락을 만들 때 어떤 하드웨어와 보안 기능이 필요한가요?

[1] SQL 생성 중...
    SELECT
    p.guideline_name AS parent_guideline_name,
    p.domain,
    p.description AS parent_description,
    c.control_name,
    c.control_category,
    c.hardware_or_security,
    c.implementation_summary,
    c.hardware_design_point,
    c.security_purpose,
    c.easy_explanation
FROM "1_parent_iot_common_guidelines" p
JOIN "2_child_home_iot_controls" c
    ON c.parent_guideline_id = p.guideline_id
WHERE (c.applicable_scope ILIKE '%도어락%'
    OR c.control_name ILIKE '%도어락%'
    OR c.description ILIKE '%도어락%'
    OR c.easy_explanation ILIKE '%도어락%'
    OR c.control_name ILIKE '%접근권한%'
    OR c.control_name ILIKE '%인증%'
    OR c.control_name ILIKE '%암호화%'
    OR c.control_name ILIKE '%보안%'
    OR c.hardware_or_security IN ('하드웨어', '보안', '통합'))
ORDER BY p.guideline_id, c.home_control_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


스마트 도어락을 만들 때는  
**문을 여닫는 하드웨어**와 **아무나 못 건드리게 하는 보안 기능**을 같이 생각해야 합니다.

## 1. 필요한 하드웨어

### 1) 문을 제어하는 핵심 장치
- **MCU(기기 동작을 조정하는 작은 컴퓨터)**  
  도어락의 전체 동작을 관리합니다.  
  예를 들면 잠금, 해제, 인증 확인, 상태 기록을 맡습니다.

- **잠금 구동부**  
  문을 실제로 잠그고 여는 부분입니다.  
  사용자 인증이 끝나면 이 부분이 동작합니다.

### 2) 사용자 확인 장치
- **인증 입력 장치**  
  비밀번호, 카드, 앱 요청 같은 사용자의 접근 요청을 받는 역할입니다.  
  자료에는 구체적인 입력 방식은 없지만, **사용자 확인 경로**가 필요하다고 볼 수 있습니다.

- **상태 확인용 센서나 감지 기능**  
  문이 닫혔는지, 열렸는지, 조작이 있었는지 확인하는 데 쓰입니다.  
  도어락은 물리적으로 만지는 제품이라 이런 확인이 중요합니다.

### 3) 통신 기능
- **원격 관리용 통신 기능**  
  앱이나 외부 관리 기능이 있다면 기기와 다른 장치가 통신해야 합니다.  
  이때 **기기끼리 진짜 장치인지 확인하는 기능**이 필요합니다.

- **암호화 처리용 기능**  
  데이터를 그냥 보내지 않고 보호해서 보내려면,  
  MCU가 직접 처리할지, 별도 보안 모듈이 맡을지 생각해야 합니다.

### 4) 보안 보조 장치
- **하드웨어 보안 모듈**  
  암호키(잠금과 해제를 보호하는 열쇠 같은 정보)를 안전하게 보관하는 용도입니다.  
  일반 저장공간보다 훨씬 안전하게 다룰 수 있습니다.

- **내부 연결 통로**  
  MCU와 보안 모듈을 I2C, UART, SPI 같은 방식으로 연결할 수 있습니다.  
  자료에서는 이런 방식으로 연결하는 구성을 고려하라고 나옵니다.

---

## 2. 연결 및 구성

### 1) 인증과 잠금 동작 연결
- 사용자가 인증 정보를 넣으면 MCU가 확인합니다.
- 확인이 끝나면 MCU가 잠금 구동부를 움직입니다.
- 이때 **허가된 사용자만 접근하도록 제한**해야 합니다.  
  아무나 설정을 바꾸지 못하게 하는 것이 핵심입니다.

### 2) 장치 간 통신 연결
- 도어락이 앱, 서버, 다른 기기와 통신한다면  
  **서로 진짜 장치인지 확인하는 절차**가 필요합니다.
- 가짜 장치가 끼어들면 문 제어 정보가 위험해질 수 있습니다.
- 그래서 **상호인증**을 넣어야 합니다.  
  상호인증은 서로가 믿을 수 있는 장치인지 확인하는 방식입니다.

### 3) 데이터 저장과 전송 구분
- 인증정보, 제어정보, 상태정보는 중요도가 높습니다.
- 이 정보가 저장되는 메모리와 밖으로 나가는 통신 경로를 따로 살펴야 합니다.
- 중요한 데이터는 **안전한 통신채널**로 보내야 합니다.  
  그래야 도청이나 중간 변조를 줄일 수 있습니다.

현재 자료에서는 도어락의 정확한 센서 종류나 핀 연결까지는 확인하기 어렵습니다.  
다만, **MCU 중심으로 인증 입력, 잠금 구동부, 통신 기능, 보안 모듈을 연결하는 구조**가 필요하다고 볼 수 있습니다.

---

## 3. 보안 설정

### 1) 외부 연결 포트 보호
- USB, RS232, Ethernet, SD Card 같은 포트는  
  운영에 필요하지 않다면 **비활성화하거나 제거**하는 것이 좋습니다.
- 이유는 외부에서 기기에 직접 꽂아 접근하는 공격을 줄이기 위해서입니다.
- 도어락은 집 안팎에서 만질 수 있으므로, 물리 포트가 위험 통로가 될 수 있습니다.

### 2) 내부 디버깅 기능 보호
- UART, JTAG(기기 내부를 점검하는 연결 통로) 같은 내부 점검용 포트는  
  완제품에서는 제거하거나 비활성화해야 합니다.
- 이유는 펌웨어를 빼내거나 내부 정보에 접근하는 것을 막기 위해서입니다.
- 개발할 때는 편하지만, 제품으로 나가면 공격자가 악용할 수 있습니다.

### 3) 인증 및 접근 제한
- 초기 인증정보는 반드시 바꾸어야 합니다.
- 허가된 사용자만 설정 변경이나 원격 제어를 할 수 있게 해야 합니다.
- 이유는 도어락 설정이 바뀌면 곧바로 보안 사고로 이어질 수 있기 때문입니다.

### 4) 암호화와 키 관리
- 저장 데이터와 전송 데이터는 암호화해야 합니다.
- 안전한 암호 알고리즘을 써야 하고, 암호키도 안전하게 관리해야 합니다.
- 키는 가능한 한 **안전한 영역이나 하드웨어 보안 모듈**에 저장하는 것이 좋습니다.
- 이유는 암호를 푸는 열쇠가 새면 보호 기능 전체가 무너지기 때문입니다.

### 5) 통신 보호
- 기기와 서버, 기기와 앱 사이의 통신은 안전한 통신채널을 써야 합니다.
- 이유는 누군가 내용을 엿보거나 바꾸는 것을 막기 위해서입니다.
- 특히 도어락은 잠금 해제와 연결되므로 통신 보호가 매우 중요합니다.

### 6) 업데이트와 취약점 관리
- 사용한 OS, 라이브러리, 통신 모듈은 최신 보안패치를 확인해야 합니다.
- 안전한 업데이트 경로도 필요합니다.
- 업데이트 파일이 진짜인지 확인한 뒤 설치해야 합니다.
- 이유는 가짜 업데이트가 들어오면 기기가 위험해질 수 있기 때문입니다.

### 7) 외부 조작과 분해 방지
- 케이스를 뜯거나 외부에서 조작한 흔적을 확인할 수 있어야 합니다.
- 이유는 도어락이 실제로 손으로 만지는 제품이라  
  억지 분해나 내부 조작 시도를 대비해야 하기 때문입니다.

### 8) 로그 기록
- 잠금, 해제, 설정 변경, 인증 실패 같은 중요한 동작은 기록해야 합니다.
- 이유는 문제가 생겼을 때 원인을 추적할 수 있기 때문입니다.

---

## 4. 최종 정리

스마트 도어락을 만들 때는 다음을 함께 확인해야 합니다.

- **MCU가 잠금과 인증을 안정적으로 제어하는지**
- **잠금 구동부와 인증 입력 장치가 제대로 연결되는지**
- **원격 기능이 있다면 장치 간 인증과 안전한 통신이 들어가는지**
- **외부 포트와 내부 디버깅 통로가 불필요하게 열려 있지 않은지**
- **암호키를 안전한 곳에 보관하는지**
- **업데이트와 취약점 관리를 계속할 수 있는지**
- **분해나 조작을 감지할 수 있는지**
- **중요 동작이 로그로 남는지**

한마디로 정리하면,  
**도어락은 “문을 여는 장치”가 아니라 “문을 안전하게 통제하는 장치”로 설계해야 합니다.**  

원하시면 다음 단계로  
**스마트 도어락용 하드웨어 구성도 예시**를 아주 쉽게 그려서 설명해드릴 수 있습니다.


질문: ESP32-H2에서 부품 연결에 쓰는 기능과 해킹을 막는 보안 기능을 같이 알려주세요.

[1] SQL 생성 중...
    SELECT 
    f.feature_name,
    f.feature_category,
    f.module_or_interface,
    f.hardware_or_security,
    f.design_use,
    f.security_purpose,
    f.page_start
FROM "4_child_esp32h2_features" f
LEFT JOIN "1_parent_iot_common_guidelines" g
    ON f.parent_guideline_id = g.guideline_id
WHERE f.hardware_or_security IN ('하드웨어', '보안', '통합')
ORDER BY 
    CASE f.hardware_or_security
        WHEN '하드웨어' THEN 1
        WHEN '통합' THEN 2
        WHEN '보안' THEN 3
        ELSE 4
    END,
    f.page_start;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


ESP32-H2에서 부품 연결과 해킹 방지를 같이 생각하면, 핵심은 **통신용 핀과 디버그 경로를 잘 나누고**, **민감한 데이터는 암호 기능으로 보호하는 것**입니다.

## 1. 필요한 하드웨어

### ① 센서, 주변장치, 외부 메모리 연결용 기능
- **I2C Controller**  
  센서나 보안칩 같은 주변장치를 연결할 때 씁니다.  
  예를 들어 온도 센서, 문 상태 센서, 보안 관련 칩을 붙일 수 있습니다.

- **SPI Controller**  
  외부 메모리나 다른 주변장치를 연결할 때 씁니다.  
  펌웨어나 데이터가 외부 Flash에 들어갈 수 있으므로, 연결만 하지 말고 보호도 같이 생각해야 합니다.

- **UART Controller**  
  직렬 통신이나 디버그 연결에 씁니다.  
  개발할 때는 편하지만, 제품이 완성되면 외부에서 쉽게 붙을 수 있어 위험할 수 있습니다.

- **IO MUX and GPIO Matrix**  
  GPIO(일반 입출력 핀)의 기능을 바꾸고 신호를 원하는 핀으로 보내는 데 씁니다.  
  즉, 어떤 핀을 센서 입력으로 쓸지, 어떤 핀을 통신용으로 쓸지 정할 때 중요합니다.

- **GPIO Hysteresis**  
  입력 신호가 흔들릴 때 오동작을 줄이는 데 도움이 됩니다.  
  문 열림 센서처럼 값이 불안정할 수 있는 입력에 유용합니다.

- **GPIO Power Supply Management**  
  GPIO 전원 도메인(전원 구역)과 Sleep Wake(절전 후 깨우기)를 관리합니다.  
  배터리 제품이라면 필요한 핀만 살아 있도록 해서 전력을 아낄 수 있습니다.

- **Low-Power Management**  
  주변장치를 필요한 만큼만 켜고 끄는 데 씁니다.  
  배터리로 오래 가야 하는 IoT 기기에서 중요합니다.

- **Reset and Clock**  
  시스템이 멈췄을 때 다시 시작하는 경로와 동작 기준을 관리합니다.  
  이상 상태에서 안정적으로 복구되게 해 줍니다.

### ② 내부 메모리와 외부 메모리 구조
- **System and Memory**  
  ROM, SRAM, External Flash 구조를 이해하는 기능입니다.  
  어떤 데이터는 내부에 두고, 어떤 데이터는 외부 Flash에 둘지 결정할 때 필요합니다.

- **External Memory Encryption and Decryption**  
  외부 Flash에 있는 코드와 데이터를 암호화해서 보호합니다.  
  외부 메모리를 쓰면 내용이 그대로 노출될 수 있으니, 민감한 정보 보호에 중요합니다.

### ③ 보안 연산을 위한 하드웨어
- **AES Accelerator**  
  대칭키 암호화를 빠르게 처리합니다.  
  외부 Flash나 통신 데이터 보호에 쓸 수 있습니다.

- **ECC Accelerator / RSA Accelerator / ECDSA / DSA**  
  인증과 서명에 쓰입니다.  
  이 기기가 진짜 맞는지, 펌웨어가 변조되지 않았는지 확인할 때 유용합니다.

- **HMAC Accelerator**  
  메시지가 바뀌지 않았는지 확인하는 데 씁니다.  
  인증값 검증에 도움이 됩니다.

- **SHA Accelerator**  
  데이터 무결성(중간에 변조되지 않았는지) 확인에 씁니다.

- **RNG**  
  예측하기 어려운 난수를 만듭니다.  
  비밀번호, 키, 인증값 생성에 필요합니다.

---

## 2. 연결 및 구성

### 센서와 통신부품 연결
- **I2C**는 센서나 보안칩 연결용으로 보면 됩니다.  
  작은 주변장치를 여러 개 붙일 때 자주 씁니다.

- **SPI**는 외부 Flash나 빠른 주변장치 연결에 씁니다.  
  펌웨어 저장 공간이 외부에 있다면, 이 경로가 중요합니다.

- **UART**는 개발 중 로그 확인이나 장치 간 직렬 통신에 씁니다.  
  하지만 완제품에서는 외부 노출을 줄여야 합니다.

- **IO MUX and GPIO Matrix**는  
  어떤 신호를 어떤 핀으로 보낼지 정하는 역할입니다.  
  그래서 센서 입력, 통신 출력, 디버그 신호가 서로 섞이지 않게 설계하는 데 필요합니다.

### 전원과 절전 구성
- **Low-Power Management**와 **GPIO Power Supply Management**를 함께 써서  
  평소에는 필요한 부품만 켜고, 필요할 때만 깨우는 구조로 만듭니다.  
  배터리 제품에서는 이 부분이 성능만큼 중요합니다.

### 메모리 구성
- **System and Memory**를 기준으로  
  내부 메모리와 외부 Flash의 역할을 나눕니다.  
  중요한 키나 민감 정보는 가능하면 더 안전한 위치에 두는 쪽이 좋습니다.

- 외부 Flash를 쓴다면 **External Memory Encryption and Decryption**을 같이 고려해야 합니다.  
  그냥 저장하면 내용이 읽힐 수 있기 때문입니다.

현재 자료에서는 구체적인 배선 방법까지 확인하기 어렵습니다.  
하지만 어떤 부품을 어떤 용도로 연결해야 하는지는 위 기능들로 파악할 수 있습니다.

---

## 3. 보안 설정

### 외부 연결 포트 보호
- **USB Serial/JTAG Controller**는 편리하지만, 완제품에서는 외부에서 디버그가 들어오는 통로가 될 수 있습니다.  
  그래서 필요 없으면 제한하거나 끄는 것을 검토해야 합니다.

- **JTAG Signal Source Control**은 JTAG(기기 내부를 점검하는 연결 통로)를 제어합니다.  
  디버깅용 포트가 열려 있으면 내부 정보가 보일 수 있으니 막는 용도입니다.

- **Chip Boot Control**은 다운로드 모드나 부트 경로를 제어합니다.  
  공격자가 원하지 않는 방식으로 기기에 접근하는 것을 줄이는 데 필요합니다.

### 디버깅 기능 보호
- **Debug Assistant**는 비정상 메모리 접근이나 디버그 상황을 확인하는 데 쓸 수 있습니다.  
  개발에는 도움 되지만, 운영 중에는 예외 상황만 감시하도록 써야 합니다.

- **UART Controller**도 디버그용으로 자주 쓰이므로,  
  출시 후에는 불필요한 로그 출력이나 접근을 막는 것이 좋습니다.

### 인증과 접근 제한
- **Permission Control**은 읽기, 쓰기, 실행 권한을 나눠 줍니다.  
  허용되지 않은 메모리 접근이나 주변장치 접근을 막는 데 중요합니다.

- **eFuse Controller**는 보안 설정과 키를 영구적으로 관리합니다.  
  JTAG 차단, Secure Boot(검증된 코드만 부팅), 암호 설정 같은 것을 저장하는 데 쓰입니다.

### 전원 이상과 공격 대응
- **Voltage Glitch Reset**과 **Power Supply Detector**는 전원이 순간적으로 흔들릴 때를 감지합니다.  
  이런 이상은 단순 고장일 수도 있고, 보안 공격일 수도 있습니다.  
  그래서 전원 감시는 안정성과 보안을 함께 지켜 줍니다.

### 데이터 암호화와 무결성 확인
- **External Memory Encryption and Decryption**은 외부 Flash의 내용을 보호합니다.  
- **AES Accelerator**는 데이터를 빠르게 암호화하는 데 좋습니다.  
- **SHA Accelerator**, **HMAC Accelerator**는 데이터가 바뀌지 않았는지 확인하는 데 씁니다.  
- **ECC / RSA / ECDSA / DSA**는 기기 신원 확인이나 펌웨어 서명 검증에 활용됩니다.  
- **RNG**는 이 모든 보안 기능에 필요한 키와 인증값을 만들 때 필요합니다.

---

## 4. 최종 정리

ESP32-H2로 IoT 기기를 만들 때는 다음을 같이 확인하면 좋습니다.

- 센서와 주변장치는 **I2C, SPI, UART, GPIO**로 연결한다.
- 핀 배치는 **IO MUX and GPIO Matrix**로 정리한다.
- 배터리 제품이면 **Low-Power Management**와 **GPIO Power Supply Management**를 함께 본다.
- 외부 Flash를 쓴다면 **External Memory Encryption and Decryption**을 검토한다.
- 개발용 포트인 **UART, USB Serial/JTAG, JTAG**는 제품 출시 전에 꼭 점검한다.
- **eFuse Controller**로 보안 설정을 고정하고, 필요 없는 디버그 경로는 막는다.
- **Permission Control**, **AES**, **SHA**, **HMAC**, **ECC/RSA/ECDSA**로 데이터와 펌웨어를 보호한다.
- **Voltage Glitch Reset**과 **Power Supply Detector**로 전원 이상도 함께 막는다.

한마디로,  
**부품 연결은 편하게 하되, 디버그와 외부 메모리 경로는 꼭 잠그는 설계**가 중요합니다.

원하시면 다음 단계로  
**“ESP32-H2 스마트 도어락용 구성 예시”**처럼 실제 용도에 맞춰 더 쉽게 정리해드릴게요.


질문: STM32에서 기기 내부를 보호하기 위해 사용할 수 있는 기능은 무엇인가요?

[1] SQL 생성 중...
    SELECT
  t.topic_name AS feature_name,
  t.topic_type AS feature_category,
  t.interface_or_tool AS module_or_interface,
  t.hardware_or_security,
  t.applicable_scope,
  t.description,
  t.hardware_design_point,
  t.security_purpose,
  t.easy_explanation,
  t.page_start
FROM "3_child_stm32_debug_topics" AS t
WHERE t.hardware_or_security IN ('하드웨어', '통합')
ORDER BY t.page_start;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


STM32에서 기기 내부를 보호할 때는 **디버그 연결을 그대로 열어두지 않는 것**이 중요합니다.  
현재 자료에서 확인되는 기능을 기준으로 쉽게 설명드리겠습니다.

## 1. 필요한 하드웨어
기기 내부를 점검하거나 막는 데 관련된 부품과 기능은 다음과 같습니다.

- **SWD/JTAG 디버그 연결**
  - MCU 내부를 확인하거나 프로그램을 넣는 연결선입니다.
  - 개발할 때는 꼭 필요합니다.
  - 하지만 완제품에서는 외부에 그대로 노출되면 내부가 읽힐 수 있습니다.

- **ST-LINK 디버그 프로브**
  - 컴퓨터와 STM32 보드를 연결해 내부 동작을 확인하는 장비입니다.
  - 개발 중 프로그램 작성과 점검에 사용합니다.
  - 출시 제품에서는 이 연결을 어떻게 둘지 함께 검토해야 합니다.

- **STM32CubeProgrammer**
  - 메모리에 프로그램을 넣고 읽고 검증하는 도구입니다.
  - 내부 메모리 보호 설정도 함께 관리할 수 있습니다.
  - 보안 설정을 적용한 뒤 실제로 잘 막히는지 확인할 때 씁니다.

- **Reset 및 Connection Mode**
  - 기기를 다시 시작시키는 연결입니다.
  - 오류가 나도 점검 연결을 유지하는 데 도움이 됩니다.
  - 보안 설정과 함께 맞춰야 합니다.

## 2. 연결 및 구성
자료에서 보이는 구성은 이런 흐름입니다.

- **SWD 또는 JTAG로 MCU와 디버거를 연결**
  - 개발 중에는 MCU를 프로그램하고 상태를 확인합니다.
  - 내부를 점검하는 용도입니다.

- **UART 또는 SWO/SWV로 실행 로그 확인**
  - MCU가 무엇을 하는지 글자로 확인합니다.
  - 개발 중 오류를 찾는 데 좋습니다.
  - 하지만 출시 제품에서는 로그가 너무 많이 보이면 내부 정보가 새어 나갈 수 있습니다.

- **MCO 출력**
  - 클록 신호를 밖으로 내보내는 기능입니다.
  - 동작이 정상인지 확인할 때 씁니다.
  - 외부에서 내부 동작을 더 쉽게 볼 수 있으므로 제품 단계에서는 필요 여부를 따져야 합니다.

- **저전력 상태 디버깅**
  - 배터리를 아끼는 상태에서도 점검할 수 있게 해줍니다.
  - 다만 저전력 모드에서 디버그 연결이 예상치 않게 살아있을 수 있으니 확인이 필요합니다.

## 3. 보안 설정
이 부분이 핵심입니다.  
내부 보호는 “밖에서 쉽게 들어오지 못하게 하는 것”입니다.

- **디버그 포트 보호**
  - SWD/JTAG는 내부 점검 통로입니다.
  - 이 통로를 제품에서 계속 열어두면 내부 프로그램이나 상태가 노출될 수 있습니다.
  - 그래서 출시용 제품에서는 노출 여부를 반드시 검토해야 합니다.

- **옵션 바이트와 보호 설정 관리**
  - 자료에서 STM32CubeProgrammer가 옵션 바이트와 보호 설정을 관리할 수 있다고 나옵니다.
  - 이런 설정은 내부 메모리 접근을 제한하는 데 쓰입니다.
  - 쉽게 말해, “아무나 MCU 안을 읽지 못하게 잠그는 설정”입니다.

- **불필요한 디버깅 출력 제거**
  - UART, SWO/SWV 로그는 개발할 때 유용합니다.
  - 하지만 제품에서는 내부 상태가 너무 많이 드러날 수 있습니다.
  - 그래서 출시 전에 로그를 남길지, 막을지 확인해야 합니다.

- **외부 연결 포트 보호**
  - 디버그용 핀과 테스트 지점이 밖에 노출되면 접근하기 쉬워집니다.
  - 자료에서도 외부에서 접근 가능한 핀과 테스트 지점의 노출을 함께 확인하라고 되어 있습니다.
  - 즉, 보드 설계 단계에서 포트를 쉽게 만질 수 없게 하는 것이 중요합니다.

- **인증 및 접근 제한**
  - 자료에는 상세한 인증 방식은 없지만, 보호 설정을 통해 접근 자체를 제한하는 방향을 확인할 수 있습니다.
  - 누구나 연결해서 읽지 못하게 막는 개념입니다.

- **데이터 암호화**
  - 현재 자료에서는 구체적인 암호화 방법까지 확인하기 어렵습니다.
  - 다만 내부 메모리 보호와 함께 사용하면 더 안전합니다.
  - 외부에서 메모리를 읽어도 내용을 바로 알기 어렵게 하는 목적입니다.

## 4. 최종 정리
STM32 내부를 보호하려면 아래를 함께 봐야 합니다.

- 개발용 **SWD/JTAG 연결을 제품에서 어떻게 처리할지**
- **ST-LINK, UART, SWO/SWV 같은 디버그 경로를 노출할지 여부**
- **STM32CubeProgrammer로 보호 설정을 적용하고 확인할지**
- **옵션 바이트 같은 보호 기능을 제대로 설정했는지**
- **출시 후 외부에서 접근 가능한 핀과 테스트 지점이 있는지**
- **디버그 로그나 내부 정보가 밖으로 새지 않는지**

한마디로 말하면,  
**개발할 때는 내부를 잘 보이게 하고, 제품이 되면 내부를 잘 숨기는 것**이 핵심입니다.

원하시면 다음 답변에서  
**“STM32에서 내부 보호를 위한 실전 체크리스트”** 형태로 더 쉽게 정리해드릴 수 있습니다.


질문: IoT 공통 보안 가이드별로 연결된 홈가전, STM32, ESP32-H2 항목이 각각 몇 개인가요?

[1] SQL 생성 중...
    SELECT
    g.guideline_id,
    g.guideline_name,
    COUNT(DISTINCT h.home_control_id) AS home_iot_count,
    COUNT(DISTINCT s.stm32_topic_id) AS stm32_count,
    COUNT(DISTINCT e.esp32_feature_id) AS esp32_h2_count
FROM "1_parent_iot_common_guidelines" g
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = g.guideline_id
LEFT JOIN "3_child_stm32_debug_topics" s
    ON s.parent_guideline_id = g.guideline_id
LEFT JOIN "4_child_esp32h2_features" e
    ON e.parent_guideline_id = g.guideline_id
GROUP BY g.guideline_id, g.guideline_name
ORDER BY g.guideline_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


아래는 **IoT 공통 보안 가이드별로 연결된 홈가전, STM32, ESP32-H2 항목 수**입니다.

1. **IoT 장치 특성을 고려한 보안 서비스 경량화**
   - 홈가전: **0개**
   - STM32: **2개**
   - ESP32-H2: **9개**

2. **접근권한 관리·인증·종단간 통신 보안·데이터 암호화**
   - 홈가전: **4개**
   - STM32: **0개**
   - ESP32-H2: **1개**

3. **소프트웨어·하드웨어 보안기술 적용 검토**
   - 홈가전: **2개**
   - STM32: **1개**
   - ESP32-H2: **7개**

4. **민감정보 보호**
   - 홈가전: **1개**
   - STM32: **0개**
   - ESP32-H2: **1개**

5. **민감정보 운영정책 투명성 보장**
   - 홈가전: **0개**
   - STM32: **0개**
   - ESP32-H2: **0개**

6. **시큐어코딩 적용**
   - 홈가전: **1개**
   - STM32: **0개**
   - ESP32-H2: **0개**

7. **소프트웨어 취약점 점검 및 보안패치 방안 구현**
   - 홈가전: **2개**
   - STM32: **1개**
   - ESP32-H2: **0개**

8. **다양한 하드웨어 보안기법 적용**
   - 홈가전: **4개**
   - STM32: **10개**
   - ESP32-H2: **6개**

9. **Secure by Default 적용**
   - 홈가전: **0개**
   - STM32: **0개**
   - ESP32-H2: **1개**

10. **안전한 보안 프로토콜 및 파라미터 설정**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **1개**

11. **취약점 분석 및 보안패치 배포**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **0개**

12. **보안취약점 및 보호조치 공지**
    - 홈가전: **0개**
    - STM32: **0개**
    - ESP32-H2: **0개**

13. **개인정보보호정책 및 보호조치 마련**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **0개**

14. **침입탐지 및 모니터링**
    - 홈가전: **0개**
    - STM32: **0개**
    - ESP32-H2: **0개**

15. **로그기록 저장·관리**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **0개**

원하시면 다음 단계로  
**“가이드별로 어떤 장치군에 보안 항목이 많이 연결되는지”**도 쉽게 정리해드릴게요.


질문: 하드웨어와 관련된 기능들을 페이지 순서대로 보여주세요.

[1] SQL 생성 중...
    SELECT
    source,
    item_name,
    page_start,
    hardware_or_security,
    category_or_tool,
    details
FROM (
    SELECT
        'STM32' AS source,
        t.topic_name AS item_name,
        t.page_start,
        t.hardware_or_security,
        t.interface_or_tool AS category_or_tool,
        t.description AS details
    FROM "3_child_stm32_debug_topics" t
    WHERE t.hardware_or_security = '하드웨어'

    UNION ALL

    SELECT
        'ESP32-H2' AS source,
        e.feature_name AS item_name,
        e.page_start,
        e.hardware_or_security,
        e.module_or_interface AS category_or_tool,
        e.description AS details
    FROM "4_child_esp32h2_features" e
    WHERE e.hardware_or_security = '하드웨어'

    UNION ALL

    SELECT
        'Home IoT' AS source,
        c.control_name AS item_name,
        c.page_start,
        c.hardware_or_security,
        c.control_category AS category_or_tool,
        c.description AS de

페이지 순서대로, 하드웨어와 관련된 기능을 쉽게 정리해드리면 아래와 같습니다.

---

## 1. 필요한 하드웨어

### STM32 쪽
- **ST-LINK 디버그 프로브**
  - STM32 MCU와 연결해서 기기 동작을 확인하는 도구입니다.
  - 문제를 찾거나 프로그램이 제대로 들어갔는지 확인할 때 씁니다.
- **SWD/JTAG 핀 연결**
  - 보드와 디버거를 이어주는 연결선입니다.
  - 기기 내부 상태를 점검하고, 프로그램을 넣고, 오류를 찾는 데 필요합니다.
- **Reset 및 Connection Mode**
  - 기기를 다시 시작하거나 연결 방식을 바꾸는 기능입니다.
  - 멈춘 기기를 복구하거나, 디버그 연결이 안 될 때 도움을 줍니다.
- **저전력 상태 디버깅**
  - 전기를 아껴 쓰는 상태에서도 동작을 확인할 수 있게 해줍니다.
  - 배터리 제품처럼 전력 관리가 중요한 기기에 필요합니다.
- **Printf via UART**
  - UART 또는 Virtual COM Port로 실행 중 메시지를 출력하는 기능입니다.
  - 기기가 지금 어떤 상태인지 글자로 확인할 수 있습니다.
- **Printf via SWO/SWV**
  - SWD 기반으로 실행 정보를 보는 방법입니다.
  - 추가 통신선을 많이 쓰지 않고도 상태 확인에 도움이 됩니다.
- **하드웨어 핀 탐색**
  - 보드의 핀 상태를 찾아보는 기능입니다.
  - 어떤 핀이 어떤 역할을 하는지 확인할 때 유용합니다.
- **Microcontroller Clock Output(MCO)**
  - MCU의 클록 신호를 밖으로 내보내는 기능입니다.
  - 클록이 제대로 나오는지 확인할 때 사용합니다.

### ESP32-H2 쪽
- **System and Memory**
  - 내부 ROM, SRAM, 외부 Flash 구조를 다룹니다.
  - 프로그램과 데이터를 어디에 저장하고 읽을지 정하는 데 중요합니다.
- **IO MUX and GPIO Matrix**
  - GPIO 핀을 어떤 기능에 연결할지 정하는 부분입니다.
  - 센서, 통신 기능, 입력 버튼 등을 원하는 핀에 배치할 때 씁니다.
- **GPIO Hysteresis**
  - 입력 신호의 흔들림을 줄여주는 기능입니다.
  - 노이즈 때문에 잘못 눌렸다고 판단하는 일을 줄입니다.
- **GPIO Power Supply Management**
  - 핀 전원 영역을 관리합니다.
  - 저전력 상태에서도 어떤 핀이 깨어날 수 있는지 정하는 데 쓰입니다.
- **Reset and Clock**
  - 칩 전체와 주변장치의 초기화, 클록 설정을 다룹니다.
  - 기기의 안정적인 시작과 동작에 중요합니다.
- **Low-Power Management**
  - Sleep, Wake-up 같은 절전 동작을 관리합니다.
  - 배터리로 오래 써야 하는 기기에 중요합니다.
- **UART Controller**
  - 외부 기기와 직렬 통신하는 기능입니다.
  - 디버그 메시지 출력이나 다른 장치 연결에 사용합니다.
- **SPI Controller**
  - SPI 방식 주변장치나 메모리를 연결하는 통신 기능입니다.
  - 빠른 통신이 필요한 부품에 씁니다.
- **I2C Controller**
  - I2C 방식 센서나 주변장치를 연결하는 기능입니다.
  - 여러 센서를 비교적 적은 선으로 연결할 때 유용합니다.

---

## 2. 연결 및 구성

### STM32
- **ST-LINK, SWD, JTAG**
  - MCU를 개발용 장비와 연결하는 구조입니다.
  - 프로그램 다운로드와 디버깅에 사용합니다.
- **UART / VCP**
  - 실행 중 로그를 출력하는 통로입니다.
  - 기기 상태를 화면에서 확인할 때 씁니다.
- **SWO / SWV / SWD**
  - 실행 정보를 더 자세히 보는 연결 방식입니다.
  - 동작 추적과 문제 분석에 도움이 됩니다.
- **MCO**
  - 외부에서 클록 신호를 확인할 수 있게 합니다.
  - 클록 설정이 맞는지 검증할 때 사용합니다.

### ESP32-H2
- **GPIO와 IO MUX**
  - 센서나 통신선이 어떤 핀으로 들어갈지 정합니다.
  - 하드웨어를 깔끔하게 배치하는 데 중요합니다.
- **UART, SPI, I2C**
  - UART는 글자 형태의 통신에 쓰입니다.
  - SPI는 빠른 연결이 필요할 때 씁니다.
  - I2C는 센서 여러 개를 묶어 연결할 때 좋습니다.
- **외부 Flash와 메모리 구조**
  - 프로그램과 데이터를 저장하는 곳을 구성합니다.
  - 저장 공간과 실행 공간을 나눠 생각해야 합니다.
- **저전력 관련 전원 도메인**
  - 일부 핀과 기능은 잠자기 상태에서도 유지됩니다.
  - 배터리 제품에서 깨어나는 조건을 만들 때 중요합니다.

---

## 3. 보안 설정

하드웨어를 만들 때는 디버그와 통신 기능이 편리하지만, 그대로 두면 위험할 수 있습니다.

- **외부 연결 포트 보호**
  - SWD, JTAG, UART 같은 연결은 개발할 때 유용합니다.
  - 하지만 완성된 제품에서는 외부에서 함부로 접근하지 못하게 막아야 합니다.
  - 이유는 내부 프로그램이나 상태가 노출될 수 있기 때문입니다.
- **디버깅 기능 보호**
  - ST-LINK, SWD/JTAG, SWO/SWV 같은 기능은 점검용입니다.
  - 제품 출시 후에는 제한하거나 비활성화하는 것이 좋습니다.
  - 이유는 내부 동작을 들여다보는 통로가 되기 때문입니다.
- **인증 및 접근 제한**
  - 누구나 통신하거나 설정을 바꾸지 못하게 해야 합니다.
  - 개발용과 실제 사용용 권한을 나누는 것이 좋습니다.
  - 이유는 잘못된 접근으로 기기 설정이 바뀌는 것을 막기 위해서입니다.
- **데이터 암호화**
  - 외부 Flash나 통신으로 오가는 정보는 보호가 필요할 수 있습니다.
  - 이유는 저장된 정보나 전달 중인 정보가 쉽게 읽히는 것을 막기 위해서입니다.
- **저전력 상태와 보안의 연결**
  - 절전 중에도 깨어나는 핀과 디버그 기능이 있으면 관리가 필요합니다.
  - 이유는 잠자는 동안에도 원치 않는 접근이 가능할 수 있기 때문입니다.

현재 자료에서는 **구체적으로 어떤 보안 설정을 켜야 하는지**까지는 확인하기 어렵습니다.  
하지만 최소한 **개발용 디버그 포트와 사용자용 통신 포트는 분리해서 생각**하는 것이 좋습니다.

---

## 4. 최종적으로 확인할 것

하드웨어와 보안을 같이 볼 때는 아래를 확인하면 좋습니다.

- 어떤 **핀을 센서용, 통신용, 디버그용**으로 쓸지 구분하기
- **UART, SPI, I2C, SWD/JTAG** 같은 연결이 서로 겹치지 않는지 보기
- 제품 출시 후에도 **디버그 포트가 열려 있지 않은지** 확인하기
- **저전력 동작** 중에도 필요한 기능만 살아 있는지 보기
- 외부 Flash나 통신으로 가는 정보에 **보호가 필요한지** 판단하기
- 기기 상태 확인용 로그를 **개발용으로만 쓸지** 정하기

---

원하시면 다음 단계로는  
**“STM32와 ESP32-H2를 기준으로 스마트 도어락 같은 제품에 어떻게 연결하면 되는지”**  
같은 형태로 더 쉽게 풀어서 설명해드릴게요.

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] CSV 파일 준비 및 데이터 확인 완료
- [ ] CSV 데이터 탐색 및 통계 분석 완료
- [ ] Supabase 연결 완료
- [ ] CSV 데이터 업로드 완료
- [ ] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [ ] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [ ] 자연어 질문으로 SQL 생성 테스트 완료
- [ ] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [ ] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용